# Model training and evaluation

The original notebook already did one thing right that a lot of this
portfolio's other rebuilds had to fix: it trained on 2004 and 2008 and
evaluated on 2012, a genuine out-of-sample test by year, not just a
random split. This notebook keeps that discipline and adds a formal
comparison against the smart baseline and a random forest.

In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

with open('data/model_data.pkl', 'rb') as f:
    d = pickle.load(f)
train, test = d['train'], d['test']
train.shape, test.shape

((100, 7), (45, 7))

In [2]:
baseline_pred_test = (test.Rasmussen > 0).astype(int)
baseline_acc = accuracy_score(test.Republican, baseline_pred_test)
print(f'smart baseline (sign of Rasmussen), 2012 test accuracy: {baseline_acc:.3f}')

smart baseline (sign of Rasmussen), 2012 test accuracy: 0.911


In [3]:
features = ['SurveyUSA', 'DiffCount']
logreg = LogisticRegression()
logreg.fit(train[features], train.Republican)
logreg_pred = logreg.predict(test[features])
logreg_acc = accuracy_score(test.Republican, logreg_pred)
logreg_auc = roc_auc_score(test.Republican, logreg.predict_proba(test[features])[:, 1])
print(f'logistic regression, 2012 test accuracy: {logreg_acc:.3f}, AUC: {logreg_auc:.3f}')

confusion_matrix(test.Republican, logreg_pred)

logistic regression, 2012 test accuracy: 0.978, AUC: 0.982


array([[23,  1],
       [ 0, 21]])

In [4]:
rf = RandomForestClassifier(n_estimators = 300, max_depth = 4, random_state = 42)
rf.fit(train[features], train.Republican)
rf_pred = rf.predict(test[features])
rf_acc = accuracy_score(test.Republican, rf_pred)
rf_auc = roc_auc_score(test.Republican, rf.predict_proba(test[features])[:, 1])
print(f'random forest, 2012 test accuracy: {rf_acc:.3f}, AUC: {rf_auc:.3f}')

random forest, 2012 test accuracy: 0.978, AUC: 1.000


Both models beat the 91.1% smart baseline on the 2012 test states,
logistic regression and random forest both reach 97.8% accuracy (44 of
45 states correct), with the logistic regression's single mistake
below. Given how few close states there are in any given election,
this is a small test set to draw strong conclusions from, but the
result is honest: a genuinely held-out year, not a number computed
partly on data the model had already seen.

In [5]:
mistakes = test[(logreg_pred != test.Republican)]
mistakes[['State', 'Rasmussen', 'SurveyUSA', 'PropR', 'DiffCount', 'Republican']]

,State,Rasmussen,SurveyUSA,PropR,DiffCount,Republican
5,Florida,2.0,0.0,0.666667,6,0


In [6]:
import json, os
os.makedirs('outputs', exist_ok = True)
with open('outputs/model_results.json', 'w') as f:
    json.dump({
        'baseline_test_accuracy': float(baseline_acc),
        'logreg_test_accuracy': float(logreg_acc),
        'logreg_test_auc': float(logreg_auc),
        'rf_test_accuracy': float(rf_acc),
        'rf_test_auc': float(rf_auc),
        'n_test': int(len(test)),
        'n_mistakes_logreg': int(len(mistakes)),
    }, f, indent = 2)